# Lab 6: AgentCore Deployment Patterns
## NovaPay Fraud Detection & Payment Support

**Objective:** Understand AWS Bedrock AgentCore architecture and deployment patterns.
Learn how to take your local Strands agent to production with managed runtime, gateway, and identity.

**Duration:** ~40 minutes
**Prerequisites:** Labs 1–5 completed
**Model:** Claude Haiku via Amazon Bedrock (us-east-1)

---

## 0. The NovaPay Situation — Why This Lab Exists

### The customer

NovaPay is a cross-border payments company operating across West Africa, moving money for
retail customers and small businesses in NGN, GHS and several other currencies. Their support
organisation handles two very different kinds of contact:

1. **Routine payment enquiries** — "what's my balance", "did TXN-103 go through", "why was
   my transfer delayed". High volume, low complexity, repetitive.
2. **Fraud investigations** — a transaction gets flagged, and someone has to work out whether
   it is genuinely fraudulent or a false positive. Lower volume, high stakes, slow.

### The customer problem

Support agents were spending the majority of their time on category 1 — the routine questions —
which meant category 2 sat in a queue. A flagged transaction that should be reviewed in minutes
was taking hours. In payments, that delay is expensive in both directions: real fraud keeps
moving while you wait, and false positives mean a legitimate customer is locked out of their
own money and calling you angry about it.

Meanwhile the support team could not scale linearly with transaction volume. Growth meant
hiring, and hiring meant onboarding time NovaPay did not have.

### The business problem

- **Cost per contact** was flat — every enquiry consumed human time regardless of how trivial it was.
- **Fraud response time** was the real risk. Slow triage means higher fraud losses and higher
  customer churn from false positives.
- **Coverage gaps** — no 24/7 support, but payments do not stop at 5pm.
- **Inconsistency** — two agents given the same flagged transaction would reach different
  conclusions, which is a problem when a regulator asks you to justify a decision.

### Why an agent was proposed

An AI agent with tool access can answer the routine questions instantly and perform the
first-pass fraud triage consistently — pulling the transaction, scoring the risk signals,
and either resolving it or escalating with the evidence already assembled. That frees the
human team to work the genuinely ambiguous cases.

Labs 1–5 built that agent. **This lab is about the part that actually makes it real.**

### Why *this* lab specifically

A Strands agent running in a Jupyter notebook is a prototype, not a product. Before NovaPay can
put it in front of customers it needs:

| Requirement | Why NovaPay needs it | AgentCore component |
|---|---|---|
| Somewhere to run that scales | Payment volume spikes on paydays and month-end | **Runtime** |
| A governed way for systems to call it | The mobile app and internal tools must integrate | **Gateway** |
| Per-user authentication and authorization | Not every caller may run a fraud investigation | **Identity** |
| Conversation state per customer | Amara's context must never leak into Kwame's session | **Memory** |
| Tracing, metrics and logs | Regulators ask "why did the agent do that?" | **Observability** |

Building all five of those yourself is months of infrastructure work. AgentCore provides them
as a managed platform — which is the proposal that was put to the customer, and what this
lab walks through.

---

## AgentCore Components

| Component | Purpose |
|---|---|
| **Runtime** | Managed compute for agent execution |
| **Gateway** | API management + MCP server endpoints |
| **Identity** | Authentication & authorization |
| **Memory** | Cross-session state persistence |
| **Observability** | Tracing, metrics, and logging |

In [ ]:
%pip install strands-agents==1.2.0 strands-agents-tools==0.2.3 boto3>=1.34.0 --quiet

## 1. Setup

In [ ]:
import json
import os
from datetime import datetime
from IPython.display import HTML, display

from strands import Agent, tool
from strands.models import BedrockModel
from botocore.config import Config as BotocoreConfig

REGION = "us-east-1"

bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    boto_client_config=BotocoreConfig(
        retries={"max_attempts": 3},
        connect_timeout=5,
        read_timeout=60
    )
)

display(HTML('<h3 style="color: #2ecc71;">✅ Setup complete</h3>'))

## 2. What Is AgentCore?

AgentCore is AWS's managed platform for deploying AI agents to production.
Think of it as **"ECS for agents"** — it handles the infrastructure so you focus on agent logic.

```
┌─────────────────────────────────────────────────────────┐
│                  AWS BEDROCK AGENTCORE                   │
│                                                           │
│  ┌─────────────┐  ┌──────────────┐  ┌─────────────────┐  │
│  │   RUNTIME   │  │   GATEWAY    │  │    IDENTITY     │  │
│  │ • Container │  │ • API Mgmt   │  │ • OAuth/OIDC    │  │
│  │ • Auto-scale│  │ • MCP Servers│  │ • Token Auth    │  │
│  │ • Health    │  │ • Rate Limit │  │ • Per-user      │  │
│  └─────────────┘  └──────────────┘  └─────────────────┘  │
│                                                           │
│  ┌─────────────┐  ┌──────────────────┐                   │
│  │   MEMORY    │  │  OBSERVABILITY   │                   │
│  │ • Sessions  │  │ • Traces         │                   │
│  │ • Long-term │  │ • Metrics        │                   │
│  │ • Per-user  │  │ • Logs           │                   │
│  └─────────────┘  └──────────────────┘                   │
└─────────────────────────────────────────────────────────┘
```

### Why AgentCore?

- **No infra management**: No ECS tasks, ALBs, or VPC configuration
- **Built-in scaling**: Handles traffic spikes automatically
- **Integrated observability**: Traces every agent step
- **Session isolation**: Each user gets their own memory space

## 3. Deployment Architecture

The journey from local development to production:

```
   LOCAL DEVELOPMENT                    AGENTCORE PRODUCTION
   ┌────────────────────┐               ┌──────────────────────────┐
   │  Jupyter/Local     │               │    AgentCore Runtime     │
   │                    │   Deploy      │  ┌────────────────────┐  │
   │  agent = Agent(    │  ─────────▶   │  │   Your Agent Code  │  │
   │      model=...,    │               │  │   (containerized)  │  │
   │      tools=[...]   │               │  └────────────────────┘  │
   │  )                 │               │  ┌────────────────────┐  │
   │                    │               │  │   MCP Tool Servers │  │
   │  agent(query)      │               │  └────────────────────┘  │
   └────────────────────┘               └──────────────────────────┘
```

### Key Principle

Your Strands agent code stays **almost identical** between local and production.
AgentCore wraps it with infrastructure (networking, auth, scaling).

## 4. Agent Code Structure for AgentCore

Here's the template structure that deploys to AgentCore Runtime.

In [ ]:
# === AgentCore Deployment Template ===
# This is the structure your agent code follows for AgentCore deployment

# File: agent.py (entry point)
AGENTCORE_TEMPLATE = '''
import json
from strands import Agent, tool
from strands.models import BedrockModel
from botocore.config import Config as BotocoreConfig

# ---- Tool Definitions ------------------------------------------------
@tool
def check_balance(customer_id: str) -> str:
    """Check NovaPay account balance.
    Args:
        customer_id: Customer ID
    """
    # In production: call DynamoDB or your data layer
    pass

@tool
def fraud_check(transaction_id: str) -> str:
    """Assess fraud risk for a transaction.
    Args:
        transaction_id: Transaction to analyze
    """
    # In production: call your ML model or rules engine
    pass

# ---- Model Configuration ---------------------------------------------
model = BedrockModel(
    model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    boto_client_config=BotocoreConfig(
        retries={"max_attempts": 3},
        connect_timeout=5,
        read_timeout=60
    )
)

# ---- Agent Definition ------------------------------------------------
agent = Agent(
    model=model,
    tools=[check_balance, fraud_check],
    system_prompt="""You are NovaPay\'s production support agent..."""
)

# ---- Handler (AgentCore entry point) ---------------------------------
def handler(event, context):
    """AgentCore invokes this function for each request."""
    user_message = event.get("message", "")
    session_id = event.get("session_id", "default")

    # Agent processes the request
    response = agent(user_message)

    return {
        "response": str(response),
        "session_id": session_id
    }
'''

print("📄 AgentCore Deployment Template:")
print("=" * 50)
print(AGENTCORE_TEMPLATE)
print("\n✅ This template shows the structure. Key parts:")
print("   1. Tool definitions (@tool decorated functions)")
print("   2. Model configuration (BedrockModel)")
print("   3. Agent creation (Agent with tools + system prompt)")
print("   4. Handler function (entry point for AgentCore)")

## 5. Session Management — Isolate Per User

In production, each user needs their own conversation context.
AgentCore provides built-in session management.

### Session Isolation Pattern

```
  User A (session-a-001) ──┐
                            ├──▶ AgentCore Runtime ──▶ Shared Agent Code
  User B (session-b-001) ──┘              │
                                          ▼
                                  ┌─────────────────┐
                                  │  Session Store  │
                                  │ session-a-001:  │
                                  │      [msgs]     │
                                  │ session-b-001:  │
                                  │      [msgs]     │
                                  └─────────────────┘
```

In [ ]:
# Session management pattern for AgentCore

class AgentCoreSessionManager:
    """Demonstrates session isolation pattern used in AgentCore.
    In production, this is backed by DynamoDB or AgentCore's built-in memory.
    """

    def __init__(self):
        self._sessions = {}  # In production: DynamoDB table

    def get_or_create_session(self, session_id: str, user_id: str) -> dict:
        """Get existing session or create new one."""
        if session_id not in self._sessions:
            self._sessions[session_id] = {
                "session_id": session_id,
                "user_id": user_id,
                "created_at": datetime.now().isoformat(),
                "messages": [],
                "metadata": {}
            }
        return self._sessions[session_id]

    def add_message(self, session_id: str, role: str, content: str):
        """Append a message to session history."""
        if session_id in self._sessions:
            self._sessions[session_id]["messages"].append({
                "role": role,
                "content": content,
                "timestamp": datetime.now().isoformat()
            })

    def get_context(self, session_id: str, window: int = 10) -> list:
        """Get recent messages for context."""
        if session_id in self._sessions:
            return self._sessions[session_id]["messages"][-window:]
        return []


# Demo: Two users with isolated sessions
session_mgr = AgentCoreSessionManager()

# User A's session
session_a = session_mgr.get_or_create_session("sess-amara-001", "CUST-1001")
session_mgr.add_message("sess-amara-001", "user", "Check my balance")
session_mgr.add_message("sess-amara-001", "assistant", "Your balance is 15,420.50 NGN")

# User B's session (completely isolated)
session_b = session_mgr.get_or_create_session("sess-kwame-001", "CUST-1002")
session_mgr.add_message("sess-kwame-001", "user", "Is TXN-103 complete?")
session_mgr.add_message("sess-kwame-001", "assistant", "Yes, TXN-103 for 150 GHS to Accra Utilities is complete.")

print("✅ Session isolation demo:")
print(f"\n👤 User A (Amara) context: {session_mgr.get_context('sess-amara-001')}")
print(f"\n👤 User B (Kwame) context: {session_mgr.get_context('sess-kwame-001')}")
print("\n💡 Each user only sees their own conversation history.")

## 6. Gateway — MCP Servers as Managed Endpoints

AgentCore Gateway turns your tools into managed API endpoints.
External systems connect to your agent via **MCP (Model Context Protocol)** servers.

```
┌────────────────────────────────────────────┐
│            AGENTCORE GATEWAY                │
│                                              │
│   External Client ──▶ API Endpoint          │
│                            │                 │
│                            ▼                 │
│                    ┌───────────────┐         │
│                    │  MCP Server   │         │
│                    │  • Auth       │         │
│                    │  • Rate Limit │         │
│                    │  • Logging    │         │
│                    └───────┬───────┘         │
│                            ▼                 │
│                    ┌───────────────┐         │
│                    │  Your Agent   │         │
│                    └───────────────┘         │
└────────────────────────────────────────────┘
```

### MCP Server Benefits

- **Standardized interface**: Any MCP-compatible client can call your agent
- **Built-in auth**: Token validation at the gateway level
- **Rate limiting**: Protect your agent from abuse
- **Tool discovery**: Clients can list available tools dynamically

In [ ]:
# MCP Server configuration structure (what you'd define for AgentCore Gateway)

MCP_SERVER_CONFIG = {
    "server_name": "novapay-support-agent",
    "description": "NovaPay customer support and fraud detection agent",
    "version": "1.0.0",
    "tools": [
        {
            "name": "check_balance",
            "description": "Check NovaPay account balance for a customer",
            "input_schema": {
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string", "description": "Customer ID"}
                },
                "required": ["customer_id"]
            }
        },
        {
            "name": "fraud_check",
            "description": "Assess fraud risk for a transaction",
            "input_schema": {
                "type": "object",
                "properties": {
                    "transaction_id": {"type": "string", "description": "Transaction ID"}
                },
                "required": ["transaction_id"]
            }
        }
    ],
    "auth": {
        "type": "oauth2",
        "token_endpoint": "https://auth.novapay.example.com/token",
        "scopes": ["agent:invoke", "tools:read"]
    },
    "rate_limits": {
        "requests_per_minute": 60,
        "requests_per_hour": 1000
    }
}

print("📄 MCP Server Configuration for AgentCore Gateway:")
print(json.dumps(MCP_SERVER_CONFIG, indent=2))

## 7. Identity — Delegated Authentication

AgentCore uses **token-based auth** (not role strings).
This follows the principle of least privilege — agents get scoped tokens, not broad IAM roles.

### Auth Flow

```
1. Client authenticates with Identity Provider (Cognito, Okta, etc.)
2. Client receives scoped access token
3. Client calls AgentCore Gateway with token in Authorization header
4. Gateway validates token and extracts user context
5. Agent receives user context (user_id, permissions) — NOT the raw token
```

### Why Token-Based (Not Role Strings)?

- **Scoped**: Token carries exactly what the user can do
- **Expirable**: Tokens expire (roles don't)
- **Auditable**: Every action traces back to a specific token issuance
- **Delegated**: Agent doesn't need to implement auth logic

In [ ]:
# Identity pattern: how the agent receives user context from AgentCore

def agent_handler_with_identity(event: dict, context: dict) -> dict:
    """
    AgentCore handler that receives user identity from the Gateway.
    The Gateway has already validated the token — we get clean user context.
    """
    # AgentCore injects user context after token validation
    user_context = event.get("user_context", {})
    user_id = user_context.get("user_id")          # e.g., "CUST-1001"
    permissions = user_context.get("scopes", [])   # e.g., ["balance:read", "transactions:read"]

    message = event.get("message", "")
    session_id = event.get("session_id", "")

    # Permission check before tool execution
    if "fraud:admin" not in permissions and "investigate fraud" in message.lower():
        return {
            "response": "You don't have permission to run fraud investigations.",
            "status": "unauthorized"
        }

    # Normal processing with user context available to tools
    # The agent can use user_id to scope data access
    return {
        "response": f"Processing request for user {user_id}",
        "session_id": session_id,
        "status": "ok"
    }


# Simulate a request with identity context
mock_event = {
    "message": "Check my balance",
    "session_id": "sess-001",
    "user_context": {
        "user_id": "CUST-1001",
        "name": "Amara Okafor",
        "scopes": ["balance:read", "transactions:read"]
    }
}

result = agent_handler_with_identity(mock_event, {})
print(f"✅ Handler result: {json.dumps(result, indent=2)}")

# Simulate unauthorized request
mock_event_unauthorized = {
    "message": "Investigate fraud on TXN-102",
    "session_id": "sess-001",
    "user_context": {
        "user_id": "CUST-1001",
        "name": "Amara Okafor",
        "scopes": ["balance:read", "transactions:read"]
    }
}

result2 = agent_handler_with_identity(mock_event_unauthorized, {})
print(f"\n🚫 Unauthorized result: {json.dumps(result2, indent=2)}")

## 8. Production Deployment Checklist

Use this checklist when taking your NovaPay agent to production:

In [ ]:
DEPLOYMENT_CHECKLIST = {
    "Runtime": [
        "✅ Agent code containerized (Dockerfile with dependencies)",
        "✅ Health check endpoint configured",
        "✅ Auto-scaling rules defined (min/max instances)",
        "✅ Cold start optimized (model client pre-warmed)",
        "✅ Timeout configured (match read_timeout in BotocoreConfig)"
    ],
    "Gateway": [
        "✅ MCP server configuration defined",
        "✅ Rate limiting configured per client",
        "✅ CORS settings if web client",
        "✅ API versioning strategy (v1/v2 paths)",
        "✅ Request/response logging enabled"
    ],
    "Identity": [
        "✅ OAuth2/OIDC provider integrated (Cognito, Okta)",
        "✅ Token scopes mapped to agent permissions",
        "✅ Token expiry set (recommend 1hr for interactive, 5min for API)",
        "✅ User context propagation to agent handler"
    ],
    "Memory": [
        "✅ Session store provisioned (DynamoDB)",
        "✅ TTL configured for session expiry",
        "✅ Per-user isolation enforced",
        "✅ Conversation window size set (recommend 20-50 messages)"
    ],
    "Observability": [
        "✅ Distributed tracing enabled (X-Ray or OpenTelemetry)",
        "✅ Custom metrics: latency, tool calls, token usage",
        "✅ Error alerting configured (CloudWatch Alarms)",
        "✅ Cost tracking per session/user",
        "✅ Guardrail intervention logging"
    ],
    "Security": [
        "✅ Guardrails configured (denied topics, PII)",
        "✅ Input validation on all tool parameters",
        "✅ Secrets in AWS Secrets Manager (not env vars)",
        "✅ VPC configuration if accessing internal services",
        "✅ Data encryption at rest and in transit"
    ]
}

print("📋 PRODUCTION DEPLOYMENT CHECKLIST")
print("=" * 50)
total = 0
for category, items in DEPLOYMENT_CHECKLIST.items():
    print(f"\n📌 {category} ({len(items)} items)")
    for item in items:
        print(f"   {item}")
    total += len(items)

print(f"\n{'=' * 50}")
print(f"Total checklist items: {total}")

## 9. Complete Example: NovaPay Production Agent

Here's how all the pieces fit together for a production-ready NovaPay agent:

In [ ]:
# Production-ready NovaPay agent (as it would run in AgentCore)

# Mock data (in production: DynamoDB / API calls)
NOVAPAY_CUSTOMERS = {
    "CUST-1001": {"name": "Amara Okafor", "balance": 15420.50, "currency": "NGN"},
    "CUST-1002": {"name": "Kwame Mensah", "balance": 3200.00, "currency": "GHS"}
}

NOVAPAY_TRANSACTIONS = {
    "TXN-101": {"customer_id": "CUST-1001", "amount": 2500.00, "status": "completed", "fraud_flags": []},
    "TXN-102": {"customer_id": "CUST-1001", "amount": 89000.00, "status": "flagged", "fraud_flags": ["unusual_amount", "odd_hour", "new_recipient"]},
    "TXN-103": {"customer_id": "CUST-1002", "amount": 150.00, "status": "completed", "fraud_flags": []}
}

@tool
def check_balance(customer_id: str) -> str:
    """Check account balance.

    Args:
        customer_id: Customer ID
    """
    customer = NOVAPAY_CUSTOMERS.get(customer_id)
    if not customer:
        return json.dumps({"error": "Customer not found"})
    return json.dumps({"name": customer["name"], "balance": customer["balance"], "currency": customer["currency"]})

@tool
def fraud_check(transaction_id: str) -> str:
    """Assess fraud risk.

    Args:
        transaction_id: Transaction ID
    """
    txn = NOVAPAY_TRANSACTIONS.get(transaction_id)
    if not txn:
        return json.dumps({"error": "Not found"})
    flags = txn.get("fraud_flags", [])
    score = len(flags) * 30
    return json.dumps({"transaction_id": transaction_id, "risk_score": score, "flags": flags, "level": "high" if score >= 60 else "low"})

# Production agent
production_agent = Agent(
    model=bedrock_model,
    tools=[check_balance, fraud_check],
    system_prompt="""You are NovaPay's Production Support Agent.
- Check balances and investigate fraud
- Be concise and professional
- Always cite evidence from tool results"""
)

# Simulate a production request
print("=" * 60)
print("PRODUCTION SIMULATION: Agent handling a request")
print("=" * 60)

response = production_agent("Investigate TXN-102 for fraud and tell me the risk level.")
print(f"\n🤖 Production Agent: {response}")

display(HTML('<h3 style="color: #2ecc71;">✅ Production agent simulation complete</h3>'))

## 🧠 Knowledge Check

1. What are the five components of AgentCore?
2. Why does AgentCore use token-based auth instead of IAM role strings?
3. What's the purpose of the Gateway's MCP server configuration?
4. How does session isolation work in a multi-tenant environment?
5. What changes in your Strands agent code when deploying to AgentCore vs running locally?
6. Why is a production deployment checklist important for fintech agents?

### Answers

<details>
<summary>Click to reveal</summary>

1. **Runtime** (managed compute), **Gateway** (API management + MCP endpoints), **Identity**
   (authn/authz), **Memory** (cross-session state), **Observability** (traces, metrics, logs).

2. Tokens are *scoped* (carry exactly what the user can do), *expirable* (roles don't expire),
   *auditable* (every action traces to a specific token issuance), and *delegated* (the agent
   doesn't implement auth logic itself). An IAM role string is broad, static, and tells you
   nothing about which end user triggered an action.

3. It publishes your tools as a standardised, discoverable interface so any MCP-compatible
   client can call the agent — with auth, rate limiting and logging enforced at the gateway
   rather than reimplemented in the agent.

4. Each user's conversation is keyed by a distinct `session_id` in a session store (DynamoDB or
   AgentCore memory). The agent code is shared; the state is not. Amara's context is never
   visible in Kwame's session.

5. Almost nothing. The tool definitions, model configuration and Agent construction stay the
   same. You add a `handler(event, context)` entry point that AgentCore invokes per request,
   and you read `session_id` and `user_context` from the event instead of managing them yourself.

6. In fintech the cost of a production defect is regulatory, not just operational. The checklist
   forces the non-negotiables — guardrails, PII handling, secrets management, encryption,
   per-user isolation, audit logging — to be explicit and verifiable before launch, rather than
   discovered during an incident or an audit.

</details>

## 🎉 Course Complete!

You've completed all 6 labs in the NovaPay AI Agent Training:

| Lab | Topic | Key Takeaway |
|---|---|---|
| 1 | Strands Fundamentals | Agent = Model + Tools + Loop |
| 2 | Memory & Sessions | Sliding window + file persistence |
| 3 | Bedrock Guardrails | Server-side content/PII filtering |
| 4 | Orchestration Patterns | Single → Sequential → Orchestrator → Agent-as-Tool |
| 5 | Knowledge Bases | RAG retrieval + grounded answers |
| 6 | AgentCore Deployment | Runtime + Gateway + Identity |

### What's Next?

- Build your own NovaPay agent with real data sources
- Experiment with different orchestration patterns
- Deploy a prototype to AgentCore
- Add observability and evaluation (see advanced modules)

---

*NovaPay AI Agent Training — Lab 6 Complete*
*Congratulations!* 🎓